### test_roc_auc best model choice

In [0]:
from mlflow import MlflowClient
from mlflow.entities import ViewType
experiment_id = "2305299045478060"
run = mlflow.search_runs(
    experiment_ids=experiment_id,
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=1,
    order_by=["metrics.Test_ROC_AUC DESC","attribute.start_time DESC"],
)

In [0]:
run

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Test_Recall,metrics.Train_ROC_AUC,metrics.Train_Accuracy,metrics.Test_Accuracy,metrics.Train_PR_AUC,metrics.Test_PR_AUC,metrics.Test_Precision,metrics.Test_F2_Score,metrics.Train_Recall,metrics.Train_F2_Score,metrics.Train_Precision,metrics.Test_ROC_AUC,metrics.Train_F0.5_Score,metrics.Test_F0.5_Score,params.num_workers,params.predictionCol,params.probabilityCol,params.verbose,params.features_cols,params.repartition_random_shuffle,params.labelCol,params.featuresCol,params.rawPredictionCol,params.use_gpu,params.n_estimators,params.force_repartition,params.arbitrary_params_dict,params.missing,params.enable_sparse_data_optim,tags.mlflow.databricks.cluster.id,tags.mlflow.databricks.notebookRevisionID,tags.mlflow.databricks.workspaceID,tags.mlflow.source.name,tags.mlflow.databricks.notebookPath,tags.sparkDatasourceInfo,tags.mlflow.databricks.gitRepoReference,tags.mlflow.log-model.history,tags.mlflow.databricks.notebook.commandID,tags.mlflow.databricks.webappURL,tags.mlflow.source.type,tags.mlflow.databricks.cluster.libraries,tags.mlflow.databricks.gitRepoProvider,tags.mlflow.databricks.gitRepoCommit,tags.mlflow.user,tags.mlflow.databricks.gitRepoRelativePath,tags.mlflow.databricks.workspaceURL,tags.mlflow.runName,tags.mlflow.databricks.gitRepoReferenceType,tags.mlflow.databricks.cluster.info,tags.mlflow.databricks.notebookID,tags.mlflow.databricks.gitRepoStatus,tags.mlflow.databricks.gitRepoUrl
0,092ad212b9c542798011b90c2a5e5145,2305299045478060,FINISHED,dbfs:/databricks/mlflow-tracking/2305299045478...,2023-06-14 08:42:44.762000+00:00,2023-06-14 08:46:27.500000+00:00,0.530572,0.991404,0.953137,0.787743,0.977075,0.609271,0.588621,0.541247,0.892805,0.900207,0.931087,0.821568,0.92317,0.576017,1,prediction,probability,True,[],False,label,features,rawPrediction,False,100,False,"{'seed': 42, 'features_Col': 'features', 'labe...",nan,False,0322-090354-fcp8mppn,1686732387624,4325144974049722,/Repos/minheejung@mz.co.kr/Databricks/Spark_ML...,/Repos/minheejung@mz.co.kr/Databricks/Spark_ML...,"path=dbfs:/mnt/jenny_mlops/preprocessing,versi...",master,"[{""artifact_path"":""model"",""flavors"":{""spark"":{...",9118156968932250177_6092554993551166156_90efd6...,https://seoul.cloud.databricks.com,NOTEBOOK,"{""installable"":[],""redacted"":[]}",gitHub,e07c1befec341a2e534ee92e32fb76242d9399c7,minheejung@mz.co.kr,Spark_MLOps/02_train,dbc-3d35d31c-7db9.cloud.databricks.com,spark_mlops_xgboost,branch,"{""cluster_name"":""dbdemos-mlops-end2end-shared""...",2305299045477858,unknown,https://github.com/jenny5587/Databricks.git


In [0]:
run_id = run.iloc[0]['run_id']
model_name = "spark_mlops"
model_uri = f"runs:/{run_id}/model"
registered_model_version = mlflow.register_model(model_uri, model_name)

Registered model 'spark_mlops' already exists. Creating a new version of this model...
2023/06/14 15:36:22 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: spark_mlops, version 3
Created version '3' of model 'spark_mlops'.


### model stage 변경

In [0]:
from mlflow import MlflowClient
import mlflow
client = mlflow.tracking.MlflowClient()
latest_version = client.get_latest_versions(model_name, stages=["None"])[0]
latest_version

Out[3]: <ModelVersion: creation_timestamp=1686756982272, current_stage='None', description='', last_updated_timestamp=1686804285212, name='spark_mlops', run_id='092ad212b9c542798011b90c2a5e5145', run_link='', source='dbfs:/databricks/mlflow-tracking/2305299045478060/092ad212b9c542798011b90c2a5e5145/artifacts/model', status='READY', status_message='', tags={}, user_id='minheejung@mz.co.kr', version='3'>

In [0]:
client.transition_model_version_stage(name = model_name,
                                      version=latest_version.version, 
                                      stage="Production")

Out[5]: <ModelVersion: creation_timestamp=1686756982272, current_stage='Production', description='', last_updated_timestamp=1686805105542, name='spark_mlops', run_id='092ad212b9c542798011b90c2a5e5145', run_link='', source='dbfs:/databricks/mlflow-tracking/2305299045478060/092ad212b9c542798011b90c2a5e5145/artifacts/model', status='READY', status_message='', tags={}, user_id='3038864777833194', version='3'>

### 조건 후 model stage 변경 - 의사결정 조치 필요

In [0]:
if run['metrics.Test_Accuracy'][0]>=0.9 or run['metrics.Test_ROC_AUC'][0]>=0.8:
    print(f'model name : {model_name}')
    run[['run_id','experiment_id','metrics.Test_Accuracy','metrics.Test_ROC_AUC']].display()
    client = mlflow.tracking.MlflowClient()
    latest_version = client.get_latest_versions(model_name, stages=["None"])[0]
    client.transition_model_version_stage(name = model_name,version=latest_version.version, stage="Staging")
else :
    pass

model name : spark_mlops


run_id,experiment_id,metrics.Test_Accuracy,metrics.Test_ROC_AUC
092ad212b9c542798011b90c2a5e5145,2305299045478060,0.7877428998505231,0.8215680473372774
